# Phishing URL Detection

End-to-end notebook for training and evaluating a small feed-forward
classifier on lexical URL features. The model and all preprocessing
artifacts are saved to the repository root so they can be reused by
`predict.py` and the Flask web UI in `web/`.

**Architecture.** A two-hidden-layer MLP with ReLU + Dropout + L2,
trained with `binary_crossentropy` on 49 standardized features.

> This notebook is a thin wrapper around `train.py`. Run it top to
> bottom on Colab (or locally) to retrain and refresh all artifacts.


## 1. Environment setup

Mount Google Drive (Colab only) and install dependencies. On a local
machine the drive mount is skipped automatically.


In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/phishing-detection-rnn-cnn

# Install deps (Colab). Locally use:  pip install -r requirements.txt
if IN_COLAB:
    !pip install -q -r requirements.txt


## 2. Train the model

`train.py` loads `dataset_phishing.csv`, splits stratified 80/20,
fits a `StandardScaler` on the training portion, builds and trains
the MLP with early stopping, then writes:

- `my_model.keras`
- `scaler.pkl`
- `feature_names.json`
- `metrics.json`
- `history.json`


In [ ]:
!python train.py


## 3. Inspect metrics


In [ ]:
import json
from pathlib import Path

with open('metrics.json') as f:
    metrics = json.load(f)

print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"AUC-ROC   : {metrics['auc_roc']:.4f}")
print(f"Avg prec. : {metrics['average_precision']:.4f}")
print(f"Brier     : {metrics['brier_score']:.4f}")
print(f"F1        : {metrics['f1']:.4f}")
print(f"Threshold : {metrics['threshold']:.4f}")
print()
print("Confusion matrix (rows=true, cols=pred):")
print("  legit     -> ", metrics['confusion_matrix'][0])
print("  phishing  -> ", metrics['confusion_matrix'][1])


## 4. Training history


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open('history.json') as f:
    h = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h['loss'], label='train')
axes[0].plot(h['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(h['accuracy'], label='train')
axes[1].plot(h['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 5. ROC and PR curves


In [ ]:
import json
import matplotlib.pyplot as plt

with open('metrics.json') as f:
    m = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(m['roc_curve']['fpr'], m['roc_curve']['tpr'], label=f"AUC = {m['auc_roc']:.3f}")
axes[0].plot([0, 1], [0, 1], 'k:', alpha=0.5)
axes[0].set_title('ROC curve'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(m['pr_curve']['recall'], m['pr_curve']['precision'], label=f"AP = {m['average_precision']:.3f}")
axes[1].set_title('Precision-Recall curve'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 6. Predict on custom URLs


In [ ]:
from phishing_detector import PhishingDetector

detector = PhishingDetector()
for url in [
    'https://www.google.com',
    'http://secure-account-verify-login.tk/login.html',
    'https://github.com/torvalds/linux',
]:
    p = detector.predict(url)
    print(f"{p.is_phishing:<5} p={p.probability:.4f}  {url}")


## 7. Run the web UI (optional)

A small Flask web app lives in `web/`. To serve it locally:

```bash
python -m web.app --host 0.0.0.0 --port 5000
```

Then open `http://localhost:5000` in a browser. On Colab use a
tunnel (e.g. `ngrok`) to expose the port.
